In [16]:
from langchain_core.documents import Document

In [17]:
doc = Document(
    page_content="This is a test document.",
    metadata={
        "source": "test_source.txt",
        "pages": 1,
        "author": "John Doe",
        "date_created": "2024-06-01",
    },
)
doc

Document(metadata={'source': 'test_source.txt', 'pages': 1, 'author': 'John Doe', 'date_created': '2024-06-01'}, page_content='This is a test document.')

In [18]:
import os 
os.makedirs("data/text_files", exist_ok=True)

In [19]:
sample_texts ={ "../data/text_files/python_intro.txt": """Python Programming Introduction 
                 
    Python is a high-level, interpreted programming language created by Guido van Rossum in 1991. Known for its readable, English-like syntax, Python emphasizes code readability with
  significant whitespace. It supports multiple paradigms: procedural, object-oriented, and functional programming.

  Python runs on Windows, macOS, Linux, and many other platforms. It comes pre-installed on most Unix systems.
"""}

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
print("Sample text files created successfully.")

Sample text files created successfully.


In [20]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document = loader.load()
document

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction \n\n    Python is a high-level, interpreted programming language created by Guido van Rossum in 1991. Known for its readable, English-like syntax, Python emphasizes code readability with\n  significant whitespace. It supports multiple paradigms: procedural, object-oriented, and functional programming.\n\n  Python runs on Windows, macOS, Linux, and many other platforms. It comes pre-installed on most Unix systems.\n')]

In [21]:
from langchain_community.document_loaders import DirectoryLoader
loader = DirectoryLoader("../data/text_files", glob="*.txt",loader_cls=TextLoader,loader_kwargs={"encoding": "utf-8"}, show_progress=False)
documents = loader.load()
documents

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction \n\n    Python is a high-level, interpreted programming language created by Guido van Rossum in 1991. Known for its readable, English-like syntax, Python emphasizes code readability with\n  significant whitespace. It supports multiple paradigms: procedural, object-oriented, and functional programming.\n\n  Python runs on Windows, macOS, Linux, and many other platforms. It comes pre-installed on most Unix systems.\n')]

In [22]:
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
loader = DirectoryLoader("../data/pdf_files", glob="*.pdf",loader_cls=PyMuPDFLoader, show_progress=False)
pdf_documents = loader.load()
pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-29T01:51:39+00:00', 'source': '../data/pdf_files/Turboquant paper.pdf', 'file_path': '../data/pdf_files/Turboquant paper.pdf', 'total_pages': 25, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-04-29T01:51:39+00:00', 'trapped': '', 'modDate': 'D:20250429015139Z', 'creationDate': 'D:20250429015139Z', 'page': 0}, page_content='TurboQuant: Online Vector Quantization with Near-optimal\nDistortion Rate\nAmir Zandieh\nGoogle Research\nzandieh@google.com\nMajid Daliri\nNew York University\ndaliri.majid@nyu.edu\nMajid Hadian\nGoogle DeepMind\nmajidh@google.com\nVahab Mirrokni\nGoogle Research\nmirrokni@google.com\nAbstract\nVector quantization, a problem rooted in Shannon’s source coding theory, aims to quantize\nhigh-dimensional Euclidean vectors while minimizing distortion in their geometric structure. We\npropose TurboQuant to address b

In [23]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict,Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [24]:
class EmbeddingManager:
    """Handle document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model '{self.model_name}' loaded successfully.")
            print(f"Embedding dimensions: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts"""
        if not self.model:
            raise ValueError("Model not loaded.")
        try:
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise   

embedding_manager = EmbeddingManager()
embedding_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8310.99it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model 'all-MiniLM-L6-v2' loaded successfully.
Embedding dimensions: 384


In [25]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents_into_chunks(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [30]:
chunks = split_documents_into_chunks(pdf_documents)
chunks

Split 38 documents into 142 chunks

Example chunk:
Content: TurboQuant: Online Vector Quantization with Near-optimal
Distortion Rate
Amir Zandieh
Google Research
zandieh@google.com
Majid Daliri
New York University
daliri.majid@nyu.edu
Majid Hadian
Google DeepM...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-29T01:51:39+00:00', 'source': '../data/pdf_files/Turboquant paper.pdf', 'file_path': '../data/pdf_files/Turboquant paper.pdf', 'total_pages': 25, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-04-29T01:51:39+00:00', 'trapped': '', 'modDate': 'D:20250429015139Z', 'creationDate': 'D:20250429015139Z', 'page': 0}


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-29T01:51:39+00:00', 'source': '../data/pdf_files/Turboquant paper.pdf', 'file_path': '../data/pdf_files/Turboquant paper.pdf', 'total_pages': 25, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-04-29T01:51:39+00:00', 'trapped': '', 'modDate': 'D:20250429015139Z', 'creationDate': 'D:20250429015139Z', 'page': 0}, page_content='TurboQuant: Online Vector Quantization with Near-optimal\nDistortion Rate\nAmir Zandieh\nGoogle Research\nzandieh@google.com\nMajid Daliri\nNew York University\ndaliri.majid@nyu.edu\nMajid Hadian\nGoogle DeepMind\nmajidh@google.com\nVahab Mirrokni\nGoogle Research\nmirrokni@google.com\nAbstract\nVector quantization, a problem rooted in Shannon’s source coding theory, aims to quantize\nhigh-dimensional Euclidean vectors while minimizing distortion in their geometric structure. We\npropose TurboQuant to address b

In [31]:
class VectorStore:
    """Manages document embeddings in chromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()

    def _initialize_chromadb(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(name=self.collection_name,metadata={"description": "PDF Document Embeddings for RAG"})
            print(f"ChromaDB initialized with collection '{self.collection_name}' at '{self.persist_directory}'.")
            print(f"Existing Documents in Collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the collection"""
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        ids =[]
        metadatas = []
        document_texts = []
        embeddings_list = []    

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            document_texts.append(doc.page_content)

            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(ids=ids, metadatas=metadatas, documents=document_texts, embeddings=embeddings_list)
            print(f"Added {len(documents)} documents to collection '{self.collection_name}'.")
            print(f"Total Documents in Collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error adding documents to ChromaDB: {e}")
            raise

vectorstore = VectorStore()
vectorstore


ChromaDB initialized with collection 'pdf_documents' at '../data/vector_store'.
Existing Documents in Collection: 2


In [32]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-29T01:51:39+00:00', 'source': '../data/pdf_files/Turboquant paper.pdf', 'file_path': '../data/pdf_files/Turboquant paper.pdf', 'total_pages': 25, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-04-29T01:51:39+00:00', 'trapped': '', 'modDate': 'D:20250429015139Z', 'creationDate': 'D:20250429015139Z', 'page': 0}, page_content='TurboQuant: Online Vector Quantization with Near-optimal\nDistortion Rate\nAmir Zandieh\nGoogle Research\nzandieh@google.com\nMajid Daliri\nNew York University\ndaliri.majid@nyu.edu\nMajid Hadian\nGoogle DeepMind\nmajidh@google.com\nVahab Mirrokni\nGoogle Research\nmirrokni@google.com\nAbstract\nVector quantization, a problem rooted in Shannon’s source coding theory, aims to quantize\nhigh-dimensional Euclidean vectors while minimizing distortion in their geometric structure. We\npropose TurboQuant to address b

In [ ]:
### convert text chunks to embeddings
texts = [doc.page_content for doc in chunks]

### generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

### Store chunks and embeddings in vector store
vectorstore.add_documents(chunks, embeddings)

Batches: 100%|██████████| 5/5 [00:01<00:00,  2.56it/s]

Generated embeddings with shape: (142, 384)
Added 142 documents to collection 'pdf_documents'.
Total Documents in Collection: 144


In [ ]:
class RAGRetriever:
    """Retrieves relevant document chunks based on query embeddings"""

    def __init__(self, vectorstore: VectorStore, embedding_manager: EmbeddingManager):
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant document chunks for a given query"""
        print(f"Retrieving top {top_k} results for query: '{query}' with score threshold: {score_threshold}")
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score
                            "rank": i + 1
                        })
                        print(f"Retrieved Document {len(retrieved_docs)} documents (after filtering)")
                    else:
                        print(f"No Documents Found.")
                    return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return[]